Balance Dataset

In [10]:
import pandas as pd
import numpy as np
import fuzzywuzzy
from fuzzywuzzy import process
import chardet

df = pd.read_table('balance.txt', sep = ' ')

Dropping Columns in a DataFrame

In [2]:
df.drop(['Limit', 'Age'], inplace=True, axis=1)

In [3]:
df.head()

,Balance,Income,Rating,Cards,Education,Gender,Student,Married,Ethnicity
0,12.240798,14.891,283,2,11,Male,No,Yes,Caucasian
1,23.283334,106.025,483,3,15,Female,Yes,Yes,Asian
2,22.530409,104.593,514,4,11,Male,No,No,Asian
3,27.652811,148.924,681,3,11,Female,No,No,Asian
4,16.893978,55.882,357,2,16,Male,No,Yes,Caucasian


Replace values

In [3]:
df.replace('African American','African').head(10)

,Balance,Income,Rating,Cards,Education,Gender,Student,Married,Ethnicity
0,12.240798,14.891,283,2,11,Male,No,Yes,Caucasian
1,23.283334,106.025,483,3,15,Female,Yes,Yes,Asian
2,22.530409,104.593,514,4,11,Male,No,No,Asian
3,27.652811,148.924,681,3,11,Female,No,No,Asian
4,16.893978,55.882,357,2,16,Male,No,Yes,Caucasian
5,22.486178,80.180,569,4,10,Male,No,No,Caucasian
6,10.574516,20.996,259,2,12,Female,No,No,African
7,14.576204,71.408,512,2,9,Male,No,No,Asian
8,7.938090,15.125,266,5,13,Female,No,No,Caucasian
9,17.756965,71.061,491,3,19,Female,Yes,Yes,African


Grouping Data

In [4]:
grouped = df.groupby('Ethnicity')
grouped.get_group('Asian').head()

,Balance,Income,Rating,Cards,Education,Gender,Student,Married,Ethnicity
1,23.283334,106.025,483,3,15,Female,Yes,Yes,Asian
2,22.530409,104.593,514,4,11,Male,No,No,Asian
3,27.652811,148.924,681,3,11,Female,No,No,Asian
7,14.576204,71.408,512,2,9,Male,No,No,Asian
12,19.218800,80.616,394,1,7,Female,No,Yes,Asian


Store income Dataset

Dealing with inconsistent data entry

In [9]:
# Libraries
import fuzzywuzzy
from fuzzywuzzy import process
import chardet

# Set seed for reproducibility
np.random.seed()

C:\Users\ndabambil\AppData\Roaming\Python\Python314\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [5]:
income_df = pd.read_csv('store_income_data_example.csv')
income_df.head()

,id,store_name,store_email,department,income,date_measured,country
0,1,"Cullen/Frost Bankers, Inc.",NaN,Clothing,$54438554.24,14 July 2006,UK
1,2,Nordson Corporation,NaN,Tools,$41744177.01,3 December 2006,united states of america
2,3,"Stag Industrial, Inc.",NaN,Beauty,$36152340.34,12 August 2003,UNITED STATES
3,4,FIRST REPUBLIC BANK,ecanadine3@fc2.com,Automotive,$8928350.04,26 October 2006,UK
4,5,Mercantile Bank Corporation,NaN,Baby,$33552742.32,24 December 1973,UK


Text pre-processing

For this exercise, we are interested in cleaning up the "country" column to make sure there are not data entry inconsistencies in it. We could go through and check each row by hand and manually correct inconsistencies when we find them. But there's more efficient way to do this!

In [6]:
countries = income_df['country'].unique()
print(f"There are {len(countries)} unique countries")
countries


There are 34 unique countries


<StringArray>
[                      'UK ',  'united states of america',
             'UNITED STATES',                        'uk',
 ' United States of America',             'South Africa ',
            'United States.',             'United States',
             'South Africa/',            'United States ',
  'United States of America',             'South Africa.',
           'United Kingdom ', 'United States of America ',
 'United States of America/',              'south africa',
                       'UK/',           'United Kingdom.',
           ' United Kingdom',             ' South Africa',
           'United Kingdom/',              'SOUTH AFRICA',
                       ' UK',            'united kingdom',
            'UNITED KINGDOM',            ' United States',
  'UNITED STATES OF AMERICA',              'South Africa',
            'United States/',             'united states',
 'United States of America.',                        'UK',
            'United Kingdom',             

Looking at the array, we can see several inconsistencies in data entry. For example, 'United States of America/' and 'United States of America', refer to the same country, but are treated as different entries due to  differencies in punctuation. This suggests that the data entry was inconsistent.

The first thing we need to do is make everything lower case ( We can change it back at the end if we like) and remove any whitespaces at the beginning and end of cells. Inconsistencies in capitalizations and trailing white spaces are very common in text data and you can fix a good 80% of your text data entry inconsistencies by doing this.

Data inconsistencies make everything lowercase and remove any whitespaces at the beginning and end of cells.

In [7]:
# convert to lower case
income_df['country'] = income_df['country'].str.lower()

# Remove trailing white spaces
income_df['country'] = income_df['country'].str.strip()

# Let us view the data
countries = income_df['country'].unique()
print(f"There are {len(countries)} unique countries")
countries

There are 15 unique countries


<StringArray>
[                       'uk',  'united states of america',
             'united states',              'south africa',
            'united states.',             'south africa/',
             'south africa.',            'united kingdom',
 'united states of america/',                       'uk/',
           'united kingdom.',           'united kingdom/',
            'united states/', 'united states of america.',
                       'uk.']
Length: 15, dtype: str

Using fuzzywuzzy to identify strings closest in correcting columns

In [11]:
# Get the top 10 closest matches to "united kingdom"
matches = fuzzywuzzy.process.extract("uk", countries, limit=10, scorer=fuzzywuzzy.fuzz.token_sort_ratio)

# Inspect matches
matches
income_df.head()

,id,store_name,store_email,department,income,date_measured,country
0,1,"Cullen/Frost Bankers, Inc.",NaN,Clothing,$54438554.24,14 July 2006,uk
1,2,Nordson Corporation,NaN,Tools,$41744177.01,3 December 2006,united states of america
2,3,"Stag Industrial, Inc.",NaN,Beauty,$36152340.34,12 August 2003,united states
3,4,FIRST REPUBLIC BANK,ecanadine3@fc2.com,Automotive,$8928350.04,26 October 2006,uk
4,5,Mercantile Bank Corporation,NaN,Baby,$33552742.32,24 December 1973,uk


To replace all rows in our country column that have a ratio of > 90 with "uk"

In [12]:
# Function to replace rows in the provided column of the provided DataFrame
# that match the provided string above the provided ratio with the provided string
def replace_matches_in_column(df, column, string_to_match, min_ratio = 90):
    strings = df[column].unique()
    
    # Get the top 10 closest matches to our input string
    matches = fuzzywuzzy.process.extract(string_to_match, strings,
                                         limit=10, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
    
    # Only get matches with a ratio > 90
    close_matches = [matches[0] for matches in matches if matches[1] >= min_ratio]
    
    # Get the rows of all the close matches in our dataframe
    rows_with_matches = df[column].isin(close_matches)
    
    # Replace all rows with close matches with the input matches
    df.loc[rows_with_matches, column] = string_to_match
    
    # Lets us know when the function is done
    print("All done!")
    

Now that we have the function we can put it to the test

In [18]:
replace_matches_in_column(df=income_df, column='country', string_to_match="united kingdom")
replace_matches_in_column(df=income_df, column='country', string_to_match="united states")
replace_matches_in_column(df=income_df, column='country', string_to_match="united states of america")
replace_matches_in_column(df=income_df, column='country', string_to_match="south africa")
replace_matches_in_column(df=income_df, column='country', string_to_match="uk")

All done
All done
All done
All done
All done


And now let's can check the unique values in our country column again and make sure we've tidied up "uk" correctly


In [13]:
# get all the unique values in the 'country' column
countries = income_df['country'].unique()

print(f"There are {len(countries)} unique countries")
countries

There are 15 unique countries


<StringArray>
[                       'uk',  'united states of america',
             'united states',              'south africa',
            'united states.',             'south africa/',
             'south africa.',            'united kingdom',
 'united states of america/',                       'uk/',
           'united kingdom.',           'united kingdom/',
            'united states/', 'united states of america.',
                       'uk.']
Length: 15, dtype: str

In [14]:
income_df.replace('uk', 'united kingdom', inplace=True)
income_df.replace('united states of america', 'united states', inplace=True)

# Get all the unique values in the 'country' column
countries = income_df['country'].unique()

print(f"There are {len(countries)} unique countries")
countries



There are 13 unique countries


<StringArray>
[           'united kingdom',             'united states',
              'south africa',            'united states.',
             'south africa/',             'south africa.',
 'united states of america/',                       'uk/',
           'united kingdom.',           'united kingdom/',
            'united states/', 'united states of america.',
                       'uk.']
Length: 13, dtype: str

Working with Date and Time

In [16]:
# Import modules
from datetime import date

# Get the first few rows of the dataset
income_df.head()

,id,store_name,store_email,department,income,date_measured,country
0,1,"Cullen/Frost Bankers, Inc.",NaN,Clothing,$54438554.24,14 July 2006,united kingdom
1,2,Nordson Corporation,NaN,Tools,$41744177.01,3 December 2006,united states
2,3,"Stag Industrial, Inc.",NaN,Beauty,$36152340.34,12 August 2003,united states
3,4,FIRST REPUBLIC BANK,ecanadine3@fc2.com,Automotive,$8928350.04,26 October 2006,united kingdom
4,5,Mercantile Bank Corporation,NaN,Baby,$33552742.32,24 December 1973,united kingdom


Yep, there are dates in the date_measured' column. But just because we can understand that it doesn't mean that the computer understands them so.
Lets see what data types have been assigned to column

In [17]:
# Get data types for columns
income_df.dtypes

id               int64
store_name         str
store_email        str
department         str
income             str
date_measured      str
country            str
dtype: object

In [18]:
# Check the data type of our date column
income_df['date_measured'].dtype

<StringDtype(storage='python', na_value=nan)>

Convert Date Columns to datetime64
Now that we know that our date column isn't being recognized as a date, it's time to convert it so that it is recognized as a date. This is called "parsing dates" because we're taking in a string and identifying its components parts



In [19]:
# Create a new column date_parsed, with the parsed dates
income_df['date_parsed'] = pd.to_datetime(income_df['date_measured'], format='%d %B %Y')
income_df['date_parsed'].head()

0   2006-07-14
1   2006-12-03
2   2003-08-12
3   2006-10-26
4   1973-12-24
Name: date_parsed, dtype: datetime64[us]